# Statistical Analysis of Quantum Machine Learning Models

This notebook performs rigorous statistical analysis on the predictions generated by the Quantum Neural Networks (QNNs). It evaluates the predictions from models of varying depths (1 to 4) against each other and the true observed values.

The analysis follows a non-parametric workflow:
1. **Shapiro-Wilk Test:** To verify the non-normality of the prediction errors.
2. **Kruskal-Wallis H-test:** To test if the distributions of predictions across different model depths originate from the same population.
3. **Wilcoxon Signed-Rank Test:** A pairwise evaluation to determine exactly which circuit depths produce statistically distinct predictions.


In [ ]:
import os
import pennylane as qml
import tensorflow as tf
import pandas as pd
import numpy as np
from tensorflow.keras.layers import LeakyReLU

from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

In [ ]:
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
  # Restrict access of TensorFlow to specific GPU
  try:
    tf.config.experimental.set_visible_devices(gpus[0], 'GPU')
  except RuntimeError as e:
    # Visible devices must be set at program startup
    print(e)

#tf.config.set_visible_devices([], 'GPU')

In [ ]:
tf.config.experimental.get_visible_devices()

# Importing Models

## Defining the Quantum Circuit and Hybrid Model
Here we define the quantum architecture used during training. We reconstruct the model structure so we can load the pre-trained weights into it.


In [ ]:
def qnode_circuit(inputs, weights):
    # weights: (n_layers,n_qubits,3)
    n_layers = len(weights)
    n_qubits = len(weights[0])

    ###############
    # Feature Map #
    ###############
    for idx in range(n_qubits):
        # Apply Hadamard gates to create an initial superposition state
        qml.Hadamard(wires=idx)
    # Embed the classical features into the quantum state using Y-rotations
    qml.templates.AngleEmbedding(inputs, rotation='Y', wires=range(n_qubits))

    ##########
    # Ansatz #
    ##########
    for k in range(n_layers):
        # Variational Layer
        for i in range(len(weights[k])):
            # Apply parameterized rotations (the trainable weights of the circuit)
            qml.Rot(*weights[k][i],wires=i)

        # Entangling Layer
        for i in range(0, n_qubits-1):
            # Apply CNOT gates linearly across the qubits to generate entanglement
            qml.CNOT(wires=[i, i + 1])
        qml.CNOT(wires=[n_qubits-1, 0])

    ###############
    # Measurement #
    ###############
    return [qml.expval(qml.PauliZ(wires=i)) for i in range(n_qubits)]


In [ ]:
def create_quantum_model(n_qubits, weights, prev):
    input_layer = tf.keras.layers.Input(shape=(n_qubits,))

    dev = qml.device("default.qubit", wires=n_qubits)
    qnn = qml.QNode(qnode_circuit, dev, interface="tf")
    q_layer = qml.qnn.KerasLayer(qnn, weights, output_dim=n_qubits)

    activation=tf.keras.layers.Activation(tf.keras.activations.relu)
    output_layer = tf.keras.layers.Dense(len(prev), LeakyReLU(alpha=0.01))

    model = tf.keras.models.Sequential([
        input_layer
        , q_layer
        , activation
        , output_layer])

    opt = tf.keras.optimizers.Adam(learning_rate = 0.001)

    model.compile(loss=['mse'], optimizer=opt, metrics=['mae'])

    return model

## Loading Pre-Trained Weights
Instead of retraining, we define the configuration (`ansatz` and `lookback`) we wish to analyze and load the checkpointed weights for all circuit depths (1 through 4) into memory.


In [ ]:
'''
Available configurations
{"ansatz": "QNN", "lookback": 1},
{"ansatz": "QNN", "lookback": 2},
{"ansatz": "Wavelets", "lookback": 1},
{"ansatz": "Wavelets", "lookback": 2},
'''
# Hardcoded - Change for the desired configuration
ansatz = "QNN"
lookback = 1

prevision_window=[1,2,3,4,5,6]
max_layers = 4
subdir = f"Ansatz-{ansatz}-Lookback-{lookback}"

print(f"\n==============================")
print(f"Processando: Ansatz={ansatz}, Lookback={lookback}")
print(f"==============================")

if ansatz == "QNN":
# Determine the number of qubits based on the lookback window and feature set
    if lookback == 1:
        nqubits = 16
    else:
        nqubits = 32
else:
# Determine the number of qubits based on the lookback window and feature set
    if lookback == 1:
        nqubits = 86
    else:
        nqubits = 172

path_chk = os.path.abspath(os.path.join(os.getcwd(), 'checkpoint', subdir))
if not os.path.exists(path_chk):
    os.makedirs(path_chk)

all_models = {}
for i in range(max_layers):
    weight_shapes = {"weights": (i+1,nqubits,3)}
    model = create_quantum_model(nqubits, weight_shapes, prevision_window)
# Load the saved weights from the checkpoint directory
    model.load_weights(os.path.join(path_chk, f'cp-{ansatz}-lookback-{lookback}-depth-{i+1}-0100.ckpt'))
    all_models[ansatz+"-"+str(lookback)+"-"+str(i+1)] = model


# Data Loading

## Data Loading and Preprocessing
We load the test datasets (both Stoke Park and Sutherland) and apply the exact same preprocessing (cleaning, lookback shifting, and Min-Max scaling) used during the training phase.


In [ ]:
def load_table(path, prev, lookback):
    X = pd.read_csv(path)
    X.dropna(axis=0,how='any',inplace=True)
    
    # We remove all outliers from dataset
    X = X[X['EXT_PM_25'] < 1000] 

    if lookback > 1:
        for col in X.columns:
            X[col+'_Lookback'] = X.loc[:,col].shift(lookback-1)
        X = X.iloc[lookback-1:, :]

    # We copy the values in X to prepare the y dataset. The first row is removed from y 
    # since it does not have a previous value to serve as forecast
    y = X[:].drop(X.index[0])
    
    # We remove the last line in X since it doesn't have an equivalent y
    X = X.iloc[:-prev[-1],:]
    
    # We create the final y dataset by creating a new column with the predictions and 
    # removing the unnecessary information
    for i in prev:
        y[f'Prev {i} hour'] = y.loc[:,"EXT_PM_25"].shift(-(i-1))
    
    if prev[-1] == 1:
        y= y.iloc[:, -1:]
    else:
        y= y.iloc[:-(prev[-1]-1), -len(prev):]

    return X, y.values

In [ ]:
print("\nLoading Datasets\n")
path_stoke = os.path.abspath(os.path.join(os.getcwd(), 'data', 'stoke'))
path_suther = os.path.abspath(os.path.join(os.getcwd(), 'data', 'suther'))

filename_train = "wavelets-lvl5-train.csv" if ansatz == "Wavelets" else "train.csv"
filename_test  = "wavelets-lvl5-test.csv"  if ansatz == "Wavelets" else "test.csv"

train_file_stoke  = os.path.join(path_stoke, filename_train)
train_file_suther = os.path.join(path_suther, filename_train)
test_file_stoke   = os.path.join(path_stoke, filename_test)
test_file_suther  = os.path.join(path_suther, filename_test)

print(f"importing data from {train_file_stoke}")
X_train_stoke, y_train_stoke = load_table(train_file_stoke, prevision_window, lookback)
print(f"importing data from {train_file_suther}")
X_train_suther, y_train_suther = load_table(train_file_suther, prevision_window, lookback)

X_all = pd.concat([X_train_stoke, X_train_suther], axis=0)
y_all = np.vstack((y_train_stoke, y_train_suther))

#X_all = X_train_stoke
#y_all = y_train_stoke


print(f"importing data from {test_file_stoke}")
X_test_stoke, y_test_stoke = load_table(test_file_stoke, prevision_window, lookback)
print(f"importing data from {test_file_suther}")
X_test_suther, y_test_suther = load_table(test_file_suther, prevision_window, lookback)

X_test = pd.concat([X_test_stoke, X_test_suther], axis=0)
y_test = np.vstack((y_test_stoke,y_test_suther))

#X_test = X_test_stoke
#y_test = y_test_stoke


plot_time = X_test['Time'].values

In [ ]:
if lookback > 1:
    X_all = X_all.drop(["Time", "Month", "Time_Lookback", "Month_Lookback"], axis=1)
    X_test = X_test.drop(["Time", "Month", "Time_Lookback", "Month_Lookback"], axis=1)
else:
    X_all = X_all.drop(["Time", "Month"], axis=1)
    X_test = X_test.drop(["Time", "Month"], axis=1)

In [ ]:
print(f"\nThere are {X_all.shape[1]} features and {X_all.shape[0]} instances in All Train set\n")
print(X_all.head())
print(f"\nThere are {X_test.shape[1]} features and {X_test.shape[0]} instances in All Test set\n")
print(X_test.head())

In [ ]:
print("\nScaling Data\n")
scaler_x = MinMaxScaler(feature_range=(0, 1))
Xs_all  = scaler_x.fit_transform(X_all)
Xs_test = scaler_x.transform(X_test)

## Generating Predictions
We generate the full set of predictions across the test set for all loaded models (Depths 1 to 4). These arrays will form the basis of our statistical tests.


In [ ]:
all_preds = {}
for key in all_models:
    all_preds[key] = all_models[key].predict(Xs_all,verbose=1)

# Statistical Analysis

In [ ]:
from scipy import stats
path_stat = os.path.abspath(os.path.join(os.getcwd(), 'statistics', subdir))
if not os.path.exists(path_stat):
    os.makedirs(path_stat)

In [ ]:
def verify_distribution_wilcoxtest(data1, data2, p_H0):
    stat, p = stats.wilcoxon(data1, data2)
    print('Statistics=%.3f, p=%.3f' % (stat, p))
    if p > p_H0:
        print('Same distribution (fail to reject H0)')
    else:
        print('Different distribution (reject H0)')
    return stat, p

In [ ]:
def verify_distribution_shapiro(data, p_H0=0.05):
    stat, p = stats.shapiro(data)
    print(f'Statistics={stat}, p={p}')
    if p > p_H0:
        print('Probably normal distribution (fail to reject H0)')
    else:
        print('Does not follow normal distribution (reject H0)')
    return stat, p

In [ ]:
def verify_distribution_kruskal(*groups, p_H0=0.05):
    stat, p = stats.kruskal(*groups)
    print(f'Statistics={stat}, p={p}')
    for hour, value in enumerate(p):
        if value > p_H0:
            print(f'Predictions for {hour+1} hours ahead have same median (fail to reject H0)')
        else:
            print(f'Predictions for {hour+1} hours ahead do not have same median (reject H0)')
    return stat, p

## 1. Shapiro-Wilk Test for Normality
We test the null hypothesis that the data was drawn from a normal distribution. In environmental forecasting, the error distribution is typically non-normal due to extreme pollution events (spikes). A p-value < 0.05 allows us to reject the null hypothesis and justifies the use of non-parametric tests.


In [ ]:
shapiro_results = []
for key in all_models:
    model, lookback, depth = key.split("-")

    print(f"\nShapiro-Wilko test: Depth {depth}")
# Run the Shapiro-Wilk test on the predictions for each depth
    stat, p = verify_distribution_shapiro(all_preds[key], p_H0=0.05)
    shapiro_results.append([f"Depth {i+1}", stat, p])

    shapiro_df = pd.DataFrame(
        shapiro_results,
        columns=["Depth", "statistic", "p-value"]
    )
    shapiro_df.set_index("Depth")

    filename = f"shapiro-{ansatz}-lookback-{lookback}.csv"
    print(f"\nSaving Shapiro-wilko results in {os.path.join(path_stat, filename)}")
    shapiro_df.to_csv(os.path.join(path_stat, filename))

## 2. Kruskal-Wallis H-test
This is a non-parametric alternative to ANOVA. We use it to test whether the medians of all the groups (the predictions from Depths 1, 2, 3, 4, and optionally the real `y_test` values) are equal. A p-value < 0.05 indicates that at least one group is statistically distinct.


In [ ]:
kruskal_results = []

# Comparação entre todos os Depths
print("\nKruskal-Wallis analysis for all Depths")
# Compare the predictions from all depths against each other
stat, p = verify_distribution_kruskal(*all_preds.values(), p_H0=0.05)
kruskal_results.append(["All Depths", stat, p])

# Comparação Depths + valores reais
print("\nKruskal-Wallis test for all Depths + Y Dataset")
all_depths_plus_real = list(all_preds.values()) + [y_test]
# Compare all predictions AND the actual ground truth values
stat, p = verify_distribution_kruskal(*all_depths_plus_real, p_H0=0.05)
kruskal_results.append(["All Depths + Real", stat, p])

# Criar DataFrame
kruskal_df = pd.DataFrame(kruskal_results, columns=["Comparison", "statistic", "p-value"])
kruskal_df.set_index("Comparison", inplace=True)

filename = f"kruskal-{ansatz}-lookback-{lookback}.csv"
print(f"\nSaving Kruskal-Wallis results in {os.path.join(path_stat, filename)}")
kruskal_df.to_csv(os.path.join(path_stat, filename))

## 3. Wilcoxon Signed-Rank Test
Since Kruskal-Wallis only tells us that *a* difference exists, we use the Wilcoxon signed-rank test for pairwise comparisons. This generates a matrix showing the p-value between every combination of model depths (e.g., Depth 1 vs Depth 2). This proves whether adding more quantum layers genuinely changes the model's predictive behavior.


In [ ]:
wilcox_matrix = [[f"Depth {i+1}"] for i in range(max_layers)]

for i in range(max_layers):
    for ii in range(max_layers):
        if i == ii:
# The p-value of a model compared against itself is exactly 1.0
            wilcox_matrix[i].append(1)
        else:
            print(f"Wilcoxon test Depth {i+1} against Depth {ii+1}\n")
# Calculate the pairwise p-value between two specific depths
            stat, p = verify_distribution_wilcoxtest(all_preds[ansatz+"-"+lookback+"-"+str(i+1)][:,0]
                                                        ,all_preds[ansatz+"-"+lookback+"-"+str(ii+1)][:,0]
                                                        , 0.05)
            wilcox_matrix[i].append(p)
wilcox = pd.DataFrame(wilcox_matrix)
wilcox.columns = ["Index"]+[f"Depth {i+1}" for i in range(max_layers)]
wilcox.set_index('Index')

filename = f"wilcoxon_matrix-{ansatz}-lookback-{lookback}.csv"
print(f"\nSaving Wilcoxon matrix in {os.path.join(path_stat, filename)}")
wilcox.to_csv(os.path.join(path_stat, filename))